# Light training và nhận diện ảnh Flowers bên ngoài dataset

Workflow này huấn luyện nhanh VGG-16 và RepLKNet-31B trên train split của Flowers Recognition, sau đó nhận diện một ảnh hoa hoàn toàn bên ngoài dataset. Ảnh bên ngoài chỉ được dùng ở bước inference, không được đưa vào train, validation, test hoặc chọn checkpoint.

## Thiết kế chống data leakage

Dataset nội bộ vẫn dùng split manifest cố định. Ảnh mới cần đặt trong `external_images/`, khác với `data/flowers/`. Notebook kiểm tra cả cây thư mục và danh sách file trong manifest trước khi cho inference. Vì ảnh ngoài không có nhãn kiểm chứng trong dataset, đầu ra của nó được báo cáo dưới dạng dự đoán và confidence, không cộng vào Accuracy hoặc Macro-F1.

`LIGHT_EPOCHS = 1` chỉ dành cho demo nhanh; kết luận khoa học vẫn lấy từ run nhiều epoch và nhiều seed trong `Flower_Experiment_Vietnamese.ipynb`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / 'src' / 'flower_experiment.py').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('Không tìm thấy project root RepLKNet.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
DATA_DIR = PROJECT_ROOT / 'data' / 'flowers'
LIGHT_RESULTS_DIR = PROJECT_ROOT / 'results' / 'flowers_light_external_b4'
EXTERNAL_DIR = PROJECT_ROOT / 'external_images'
EXTERNAL_IMAGE_PATH = EXTERNAL_DIR / 'my_flower.jpg'
LIGHT_EPOCHS = 1
LIGHT_BATCH_SIZE = 4
LIGHT_SEED = 42
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('CUDA không khả dụng. Hãy chọn kernel Python (RepLKNet Demo).')

print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.executable}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Ảnh external dự kiến: {EXTERNAL_IMAGE_PATH}')

## 1. Kiểm tra ảnh external trước khi train/inference

Đặt ảnh cần nhận diện vào `external_images/my_flower.jpg`. Có thể đổi tên hoặc đổi biến `EXTERNAL_IMAGE_PATH`, nhưng không đặt ảnh vào `data/flowers/`.

In [ ]:
from torchvision.datasets import ImageFolder
from flower_experiment import build_transforms, locate_imagefolder_root, run_experiment

DATA_ROOT = locate_imagefolder_root(DATA_DIR)
_, EVAL_TRANSFORM = build_transforms(image_size=224)
DATASET = ImageFolder(DATA_ROOT, transform=EVAL_TRANSFORM)
MANIFEST_PATH = PROJECT_ROOT / 'results' / 'flowers' / 'split_manifest.json'
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
MANIFEST_FILES = set(MANIFEST['files'])

EXTERNAL_IMAGE_PATH = EXTERNAL_IMAGE_PATH.expanduser().resolve()
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
external_ready = EXTERNAL_IMAGE_PATH.exists()
if not external_ready:
    print('Chưa có ảnh external. Đặt file tại:')
    print(f'  {EXTERNAL_IMAGE_PATH}')
    print('Notebook vẫn có thể train nhẹ; cell inference sẽ chờ ảnh này.')
else:
    if DATA_ROOT in EXTERNAL_IMAGE_PATH.parents:
        raise RuntimeError('Ảnh external đang nằm trong data/flowers; dừng để tránh data leakage.')
    try:
        relative_name = str(EXTERNAL_IMAGE_PATH.relative_to(DATA_ROOT)).replace('\\', '/')
    except ValueError:
        relative_name = None
    if relative_name in MANIFEST_FILES:
        raise RuntimeError('Ảnh external xuất hiện trong split manifest; dừng để tránh data leakage.')
    Image.open(EXTERNAL_IMAGE_PATH).convert('RGB').verify()
    print(f'Ảnh external hợp lệ: {EXTERNAL_IMAGE_PATH}')

print(f'Dataset nội bộ: {len(DATASET)} ảnh | classes={DATASET.classes}')
print(f'Ảnh external được dùng cho train: False')

## 2. Train nhẹ trên train split

Hai model dùng cùng protocol và chỉ nhìn thấy ảnh trong `data/flowers/`. Kết quả được lưu riêng ở `results/flowers_light_external/` để không ghi đè run 3 seed đã có.

In [ ]:
LIGHT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
vgg_done = (LIGHT_RESULTS_DIR / f'seed_{LIGHT_SEED}' / 'vgg16_metrics.json').exists()
replk_done = (LIGHT_RESULTS_DIR / f'seed_{LIGHT_SEED}' / 'replknet31b_metrics.json').exists()
if vgg_done and replk_done:
    print(f'Đã có light run tại {LIGHT_RESULTS_DIR}; giữ nguyên checkpoint hiện có.')
else:
    light_summary = run_experiment(
        data_dir=DATA_DIR,
        project_root=PROJECT_ROOT,
        output_dir=LIGHT_RESULTS_DIR,
        replk_checkpoint=PROJECT_ROOT / 'weights' / 'RepLKNet-31B_ImageNet-1K_224.pth',
        epochs=LIGHT_EPOCHS,
        batch_size=LIGHT_BATCH_SIZE,
        seeds=[LIGHT_SEED],
        num_workers=0,
        device_name='cuda',
        deterministic=True,
    )
    print(f'Đã hoàn tất light training: {len(light_summary)} model-seed results')

## 3. Nạp checkpoint light và nhận diện ảnh mới

Classifier vẫn chỉ có 5 lớp Flowers. Nếu ảnh không phải hoa hoặc thuộc loại ngoài 5 lớp, confidence cao không đồng nghĩa với dự đoán đúng; đó là giới hạn open-set của bài toán này.

In [ ]:
from torchvision.models import vgg16
import torch.nn as nn
from replknet_demo import build_replknet, load_checkpoint_flexible

CLASS_NAMES = DATASET.classes
LIGHT_SEED_DIR = LIGHT_RESULTS_DIR / f'seed_{LIGHT_SEED}'
VGG_LIGHT_CHECKPOINT = LIGHT_SEED_DIR / 'vgg16_best.pt'
REPLK_LIGHT_CHECKPOINT = LIGHT_SEED_DIR / 'replknet31b_best.pt'

vgg_model = vgg16(weights=None)
vgg_model.classifier[6] = nn.Linear(vgg_model.classifier[6].in_features, len(CLASS_NAMES))
load_checkpoint_flexible(vgg_model, VGG_LIGHT_CHECKPOINT)
vgg_model = vgg_model.cpu().eval()

replk_model, _ = build_replknet(
    PROJECT_ROOT,
    model_name='RepLKNet-31B',
    checkpoint_path=None,
    device=torch.device('cpu'),
    num_classes=len(CLASS_NAMES),
    merge_for_inference=False,
)
load_checkpoint_flexible(replk_model, REPLK_LIGHT_CHECKPOINT)
replk_model.structural_reparam()
replk_model = replk_model.cpu().eval()
MODELS = {'VGG-16 (CNN thuần)': vgg_model, 'RepLKNet-31B': replk_model}

In [ ]:
if not EXTERNAL_IMAGE_PATH.exists():
    print(f'Chưa nhận diện: chưa có {EXTERNAL_IMAGE_PATH}')
else:
    external_image = Image.open(EXTERNAL_IMAGE_PATH).convert('RGB')
    external_batch = EVAL_TRANSFORM(external_image).unsqueeze(0)
    external_predictions = {}
    for model_name, model in MODELS.items():
        model = model.to(DEVICE).eval()
        with torch.inference_mode():
            probabilities = torch.softmax(model(external_batch.to(DEVICE)), dim=1)[0].cpu()
        model.cpu()
        torch.cuda.empty_cache()
        top_index = int(probabilities.argmax())
        external_predictions[model_name] = {
            'predicted_class': CLASS_NAMES[top_index],
            'confidence': float(probabilities[top_index]),
            'probabilities': probabilities.numpy(),
        }

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(external_image)
    axes[0].set_title('Ảnh external - không thuộc dataset')
    axes[0].axis('off')
    x = np.arange(len(CLASS_NAMES))
    width = 0.35
    for offset, (model_name, result) in zip((-width / 2, width / 2), external_predictions.items()):
        axes[1].bar(x + offset, result['probabilities'], width, label=model_name)
        axes[2].barh([model_name], [result['confidence']])
        print(f"{model_name}: {result['predicted_class']} ({result['confidence']:.2%})")
    axes[1].set_xticks(x, CLASS_NAMES)
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel('Xác suất')
    axes[1].set_title('Phân bố xác suất theo lớp')
    axes[1].legend()
    axes[2].set_xlim(0, 1)
    axes[2].set_xlabel('Confidence lớp dự đoán')
    axes[2].set_title('Confidence của hai model')
    plt.tight_layout()
    plt.show()

## Ghi nhận để báo cáo

Ảnh external được dùng như một phép kiểm tra ngoài mẫu (external sanity check). Không dùng kết quả của ảnh này để cập nhật trọng số, chọn epoch tốt nhất, điều chỉnh threshold hoặc tính metrics test. Nếu biết nhãn thật của ảnh, có thể ghi nhận đúng/sai riêng trong phụ lục, nhưng không gộp vào test set.